In [1]:
import pandas as pd
import numpy as np
from typing import Tuple, Dict, List
import warnings
warnings.filterwarnings('ignore')

# Configuration
N_BOOTSTRAP = 10000
CONFIDENCE_LEVEL = 0.95
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

C:\Users\alena\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\alena\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
def convert_to_user_level(df: pd.DataFrame) -> pd.DataFrame:
    user_agg = df.groupby(['user_id', 'split', 'upsell_order']).agg({
        'subscription': 'first',
        'channel': 'first',
        'geo': 'first',
        'payment_method': 'first',
        'utm_source': 'first',
        
        'ups_view': 'max',
        'ups_ttp': 'max',
        'ups_purched': 'max',
        
        'unsub12h': 'max',
        
        'purch_amount': 'sum',
        'ltv': 'mean',
        'ticket_count': 'sum',
        'is_ticket': 'max'
        
    }).reset_index()
    
    user_total_purched = df.groupby('user_id')['ups_purched'].sum().reset_index()
    user_total_purched.rename(columns={'ups_purched': 'purch_count'}, inplace=True)
    
    user_agg = user_agg.merge(user_total_purched, on='user_id', how='left')
    
    return user_agg

In [3]:
def bootstrap_difference_one_sided(
    clean: np.ndarray,
    test: np.ndarray,
    n_iterations: int = N_BOOTSTRAP,
    confidence_level: float = CONFIDENCE_LEVEL,
    metric_name: str = "mean"
) -> Dict:
    clean = np.array(clean)
    test = np.array(test)
    
    if metric_name == "median":
        metric_func = np.median
    elif metric_name == "sum":
        metric_func = np.sum
    elif metric_name == "count":
        metric_func = len
    else:
        metric_func = np.mean
    
    observed_diff = metric_func(test) - metric_func(clean)
    
    differences = []
    for _ in range(n_iterations):
        clean_sample = np.random.choice(clean, size=len(clean), replace=True)
        test_sample = np.random.choice(test, size=len(test), replace=True)
        diff = metric_func(test_sample) - metric_func(clean_sample)
        differences.append(diff)
    
    differences = np.array(differences)
    
    ci_lower = np.percentile(differences, (1 - confidence_level) * 100)
    ci_upper = np.percentile(differences, confidence_level * 100)
    
    p_value_positive = np.sum(differences <= 0) / n_iterations
    p_value_negative = np.sum(differences >= 0) / n_iterations
    p_value = min(p_value_positive, p_value_negative)
    
    return {
        'clean_mean': metric_func(clean),
        'test_mean': metric_func(test),
        'observed_difference': observed_diff,
        'ci_lower_one_sided': ci_lower,
        'ci_upper_one_sided': ci_upper,
        'p_value_one_sided': p_value,
        'significant': ci_lower > 0 or ci_upper < 0,
        'direction': 'positive' if observed_diff > 0 else 'negative',
        'clean_n': len(clean),
        'test_n': len(test)
    }

In [4]:
def calculate_conversion_rate(df: pd.DataFrame, numerator_col: str, denominator_col: str) -> np.ndarray:
    result = np.full(len(df), np.nan)
    mask = df[denominator_col] == 1
    if mask.sum() > 0:
        result[mask] = df.loc[mask, numerator_col].values
    return result

In [5]:

def analyze_ab_test(df: pd.DataFrame, variant_column: str = 'split') -> pd.DataFrame:
    variants = df[variant_column].unique()
    if len(variants) != 2:
        raise ValueError(f"Expected 2 variants, found {len(variants)}: {variants}")
    
    clean_variant = sorted(variants)[0]
    test_variant = sorted(variants)[1]
    
    print(f"clean: {clean_variant}")
    print(f"test: {test_variant}")
    print(f"clean n: {len(df[df[variant_column] == clean_variant])}")
    print(f"test n: {len(df[df[variant_column] == test_variant])}")
    print()
    
    clean_df = df[df[variant_column] == clean_variant].copy()
    test_df = df[df[variant_column] == test_variant].copy()
    
    clean_df['upsell_ttp_rate'] = calculate_conversion_rate(clean_df, 'ups_ttp', 'ups_view')
    test_df['upsell_ttp_rate'] = calculate_conversion_rate(test_df, 'ups_ttp', 'ups_view')
    
    clean_df['upsell_sr'] = calculate_conversion_rate(clean_df, 'ups_purched', 'ups_ttp')
    test_df['upsell_sr'] = calculate_conversion_rate(test_df, 'ups_purched', 'ups_ttp')
    
    clean_df['upsell_rate'] = calculate_conversion_rate(clean_df, 'ups_purched', 'ups_view')
    test_df['upsell_rate'] = calculate_conversion_rate(test_df, 'ups_purched', 'ups_view')

    clean_df['unsub_rate_view'] = calculate_conversion_rate(clean_df, 'unsub12h', 'ups_view')
    test_df['unsub_rate_view'] = calculate_conversion_rate(test_df, 'unsub12h', 'ups_view')

    clean_df['unsub_rate_purchase'] = calculate_conversion_rate(clean_df, 'unsub12h', 'ups_purched')
    test_df['unsub_rate_purchase'] = calculate_conversion_rate(test_df, 'unsub12h', 'ups_purched')
    
    results = []
    
    result = bootstrap_difference_one_sided(
        clean_df['ups_view'].values,
        test_df['ups_view'].values,
        metric_name='sum'
    )
    results.append({
        'metric': 'Upsell View Count',
        'clean_value': f"{result['clean_mean']:.0f}",
        'test_value': f"{result['test_mean']:.0f}",
        'absolute_difference': f"{result['observed_difference']:.0f}",
        'relative_difference_%': f"{(result['observed_difference'] / result['clean_mean'] * 100):.2f}" if result['clean_mean'] != 0 else 'N/A',
        'ci_lower': f"{result['ci_lower_one_sided']:.2f}",
        'ci_upper': f"{result['ci_upper_one_sided']:.2f}",
        'p_value': f"{result['p_value_one_sided']:.4f}",
        'significant': 'YES' if result['significant'] else 'NO',
        'direction': result['direction'].upper()
    })
    
    clean_ttp = clean_df[clean_df['ups_view'] == 1]['upsell_ttp_rate'].dropna().values
    test_ttp = test_df[test_df['ups_view'] == 1]['upsell_ttp_rate'].dropna().values
    
    if len(clean_ttp) > 0 and len(test_ttp) > 0:
        result = bootstrap_difference_one_sided(clean_ttp, test_ttp, metric_name='mean')
        results.append({
            'metric': 'Upsell TTP Rate (on View)',
            'clean_value': f"{result['clean_mean']:.4f}",
            'test_value': f"{result['test_mean']:.4f}",
            'absolute_difference': f"{result['observed_difference']:.4f}",
            'relative_difference_%': f"{(result['observed_difference'] / result['clean_mean'] * 100):.2f}" if result['clean_mean'] != 0 else 'N/A',
            'ci_lower': f"{result['ci_lower_one_sided']:.4f}",
            'ci_upper': f"{result['ci_upper_one_sided']:.4f}",
            'p_value': f"{result['p_value_one_sided']:.4f}",
            'significant': 'YES' if result['significant'] else 'NO',
            'direction': result['direction'].upper()
        })
    
    clean_sr = clean_df[clean_df['ups_ttp'] == 1]['upsell_sr'].dropna().values
    test_sr = test_df[test_df['ups_ttp'] == 1]['upsell_sr'].dropna().values
    
    if len(clean_sr) > 0 and len(test_sr) > 0:
        result = bootstrap_difference_one_sided(clean_sr, test_sr, metric_name='mean')
        results.append({
            'metric': 'Upsell SR (on TTP)',
            'clean_value': f"{result['clean_mean']:.4f}",
            'test_value': f"{result['test_mean']:.4f}",
            'absolute_difference': f"{result['observed_difference']:.4f}",
            'relative_difference_%': f"{(result['observed_difference'] / result['clean_mean'] * 100):.2f}" if result['clean_mean'] != 0 else 'N/A',
            'ci_lower': f"{result['ci_lower_one_sided']:.4f}",
            'ci_upper': f"{result['ci_upper_one_sided']:.4f}",
            'p_value': f"{result['p_value_one_sided']:.4f}",
            'significant': 'YES' if result['significant'] else 'NO',
            'direction': result['direction'].upper()
        })
    
    clean_rate = clean_df[clean_df['ups_view'] == 1]['upsell_rate'].dropna().values
    test_rate = test_df[test_df['ups_view'] == 1]['upsell_rate'].dropna().values
    
    if len(clean_rate) > 0 and len(test_rate) > 0:
        result = bootstrap_difference_one_sided(clean_rate, test_rate, metric_name='mean')
        results.append({
            'metric': 'Upsell Rate (on View)',
            'clean_value': f"{result['clean_mean']:.4f}",
            'test_value': f"{result['test_mean']:.4f}",
            'absolute_difference': f"{result['observed_difference']:.4f}",
            'relative_difference_%': f"{(result['observed_difference'] / result['clean_mean'] * 100):.2f}" if result['clean_mean'] != 0 else 'N/A',
            'ci_lower': f"{result['ci_lower_one_sided']:.4f}",
            'ci_upper': f"{result['ci_upper_one_sided']:.4f}",
            'p_value': f"{result['p_value_one_sided']:.4f}",
            'significant': 'YES' if result['significant'] else 'NO',
            'direction': result['direction'].upper()
        })
    
    clean_gain = clean_df[clean_df['ups_view'] == 1]['purch_amount'].values
    test_gain = test_df[test_df['ups_view'] == 1]['purch_amount'].values
    
    if len(clean_gain) > 0 and len(test_gain) > 0:
        result = bootstrap_difference_one_sided(clean_gain, test_gain, metric_name='mean')
        results.append({
            'metric': 'Upsell Gain (on View)',
            'clean_value': f"{result['clean_mean']:.2f}",
            'test_value': f"{result['test_mean']:.2f}",
            'absolute_difference': f"{result['observed_difference']:.2f}",
            'relative_difference_%': f"{(result['observed_difference'] / result['clean_mean'] * 100):.2f}" if result['clean_mean'] != 0 else 'N/A',
            'ci_lower': f"{result['ci_lower_one_sided']:.2f}",
            'ci_upper': f"{result['ci_upper_one_sided']:.2f}",
            'p_value': f"{result['p_value_one_sided']:.4f}",
            'significant': 'YES' if result['significant'] else 'NO',
            'direction': result['direction'].upper()
        })


    clean_ltv = clean_df[clean_df['ups_view'] == 1]['ltv'].values
    test_ltv = test_df[test_df['ups_view'] == 1]['ltv'].values
    
    if len(clean_ltv) > 0 and len(test_ltv) > 0:
        result = bootstrap_difference_one_sided(clean_ltv, test_ltv, metric_name='mean')
        results.append({
            'metric': 'LTV',
            'clean_value': f"{result['clean_mean']:.2f}",
            'test_value': f"{result['test_mean']:.2f}",
            'absolute_difference': f"{result['observed_difference']:.2f}",
            'relative_difference_%': f"{(result['observed_difference'] / result['clean_mean'] * 100):.2f}" if result['clean_mean'] != 0 else 'N/A',
            'ci_lower': f"{result['ci_lower_one_sided']:.2f}",
            'ci_upper': f"{result['ci_upper_one_sided']:.2f}",
            'p_value': f"{result['p_value_one_sided']:.4f}",
            'significant': 'YES' if result['significant'] else 'NO',
            'direction': result['direction'].upper()
        })
    
    clean_unsub = clean_df[clean_df['ups_view'] == 1]['unsub_rate_view'].dropna().values
    test_unsub = test_df[test_df['ups_view'] == 1]['unsub_rate_view'].dropna().values

    if len(clean_unsub) > 0 and len(test_unsub) > 0:
        result = bootstrap_difference_one_sided(clean_unsub, test_unsub, metric_name='mean')
        results.append({
            'metric': 'Unsub 12h Rate (on View)',
            'clean_value': f"{result['clean_mean']:.4f}",
            'test_value': f"{result['test_mean']:.4f}",
            'absolute_difference': f"{result['observed_difference']:.4f}",
            'relative_difference_%': f"{(result['observed_difference'] / result['clean_mean'] * 100):.2f}" if result['clean_mean'] != 0 else 'N/A',
            'ci_lower': f"{result['ci_lower_one_sided']:.4f}",
            'ci_upper': f"{result['ci_upper_one_sided']:.4f}",
            'p_value': f"{result['p_value_one_sided']:.4f}",
            'significant': 'YES' if result['significant'] else 'NO',
            'direction': result['direction'].upper()
        })
    
    clean_unsub_purchase = clean_df[clean_df['ups_purched'] == 1]['unsub_rate_purchase'].dropna().values
    test_unsub_purchase = test_df[test_df['ups_purched'] == 1]['unsub_rate_purchase'].dropna().values

    if len(clean_unsub_purchase) > 0 and len(test_unsub_purchase) > 0:
        result = bootstrap_difference_one_sided(clean_unsub_purchase, test_unsub_purchase, metric_name='mean')
        results.append({
            'metric': 'Unsub 12h Rate (on Upsell Purchase)',
            'clean_value': f"{result['clean_mean']:.4f}",
            'test_value': f"{result['test_mean']:.4f}",
            'absolute_difference': f"{result['observed_difference']:.4f}",
            'relative_difference_%': f"{(result['observed_difference'] / result['clean_mean'] * 100):.2f}" if result['clean_mean'] != 0 else 'N/A',
            'ci_lower': f"{result['ci_lower_one_sided']:.4f}",
            'ci_upper': f"{result['ci_upper_one_sided']:.4f}",
            'p_value': f"{result['p_value_one_sided']:.4f}",
            'significant': 'YES' if result['significant'] else 'NO',
            'direction': result['direction'].upper()
        })
    
    clean_ticket_users_upsell = (
        (clean_df['ups_purched'] == 1) & 
        (clean_df['ticket_count'] != 0)
    ).astype(int).values

    test_ticket_users_upsell = (
        (test_df['ups_purched'] == 1) & 
        (test_df['ticket_count'] != 0)
    ).astype(int).values

    if len(clean_ticket_users_upsell) > 0 and len(test_ticket_users_upsell) > 0:
        result = bootstrap_difference_one_sided(clean_ticket_users_upsell, test_ticket_users_upsell, metric_name='mean')
        results.append({
            'metric': 'Ticket Users Share (on All Users)',
            'clean_value': f"{result['clean_mean']:.4f}",
            'test_value': f"{result['test_mean']:.4f}",
            'absolute_difference': f"{result['observed_difference']:.4f}",
            'relative_difference_%': f"{(result['observed_difference'] / result['clean_mean'] * 100):.2f}" if result['clean_mean'] != 0 else 'N/A',
            'ci_lower': f"{result['ci_lower_one_sided']:.4f}",
            'ci_upper': f"{result['ci_upper_one_sided']:.4f}",
            'p_value': f"{result['p_value_one_sided']:.4f}",
            'significant': 'YES' if result['significant'] else 'NO',
            'direction': result['direction'].upper()
        })


    clean_ticket_users_share = clean_df[(clean_df['ups_view'] == 1) & (clean_df['ups_purched'] != 0)]['is_ticket'].dropna().values
    test_ticket_users_share = test_df[(test_df['ups_view'] == 1) & (test_df['ups_purched'] != 0)]['is_ticket'].dropna().values
    
    if len(clean_ticket_users_share) > 0 and len(test_ticket_users_share) > 0:
        result = bootstrap_difference_one_sided(clean_ticket_users_share, test_ticket_users_share, metric_name='mean')
        results.append({
            'metric': 'Ticket Users Share (on Purch Users)',
            'clean_value': f"{result['clean_mean']:.4f}",
            'test_value': f"{result['test_mean']:.4f}",
            'absolute_difference': f"{result['observed_difference']:.4f}",
            'relative_difference_%': f"{(result['observed_difference'] / result['clean_mean'] * 100):.2f}" if result['clean_mean'] != 0 else 'N/A',
            'ci_lower': f"{result['ci_lower_one_sided']:.4f}",
            'ci_upper': f"{result['ci_upper_one_sided']:.4f}",
            'p_value': f"{result['p_value_one_sided']:.4f}",
            'significant': 'YES' if result['significant'] else 'NO',
            'direction': result['direction'].upper()
        })
        
    clean_aov = clean_df[clean_df['ups_purched'] == 1]['purch_amount'].dropna().values
    test_aov = test_df[test_df['ups_purched'] == 1]['purch_amount'].dropna().values

    if len(clean_aov) > 0 and len(test_aov) > 0:
        result = bootstrap_difference_one_sided(clean_aov, test_aov, metric_name='mean')
        results.append({
            'metric': 'AOV (on Purch Users)',
            'clean_value': f"{result['clean_mean']:.2f}",
            'test_value': f"{result['test_mean']:.2f}",
            'absolute_difference': f"{result['observed_difference']:.2f}",
            'relative_difference_%': f"{(result['observed_difference'] / result['clean_mean'] * 100):.2f}" if result['clean_mean'] != 0 else 'N/A',
            'ci_lower': f"{result['ci_lower_one_sided']:.2f}",
            'ci_upper': f"{result['ci_upper_one_sided']:.2f}",
            'p_value': f"{result['p_value_one_sided']:.4f}",
            'significant': 'YES' if result['significant'] else 'NO',
            'direction': result['direction'].upper()
        })
    # Среднее количество покупок на пользователя
    clean_user_purch = clean_df.groupby('user_id')['ups_purched'].sum().values
    test_user_purch = test_df.groupby('user_id')['ups_purched'].sum().values

    if len(clean_user_purch) > 0 and len(test_user_purch) > 0:
        result = bootstrap_difference_one_sided(clean_user_purch, test_user_purch, metric_name='mean')
        results.append({
            'metric': 'Avg Purchases per User',
            'clean_value': f"{result['clean_mean']:.4f}",
            'test_value': f"{result['test_mean']:.4f}",
            'absolute_difference': f"{result['observed_difference']:.4f}",
            'relative_difference_%': f"{(result['observed_difference'] / result['clean_mean'] * 100):.2f}" if result['clean_mean'] != 0 else 'N/A',
            'ci_lower': f"{result['ci_lower_one_sided']:.4f}",
            'ci_upper': f"{result['ci_upper_one_sided']:.4f}",
            'p_value': f"{result['p_value_one_sided']:.4f}",
            'significant': 'YES' if result['significant'] else 'NO',
            'direction': result['direction'].upper()
        })
    return pd.DataFrame(results)

In [6]:
def analyze_by_segment(df: pd.DataFrame, segment_col: str, segment_name: str, 
                       variant_column: str = 'split') -> Dict[str, pd.DataFrame]:
    results = {}
    
    segment_values = df[segment_col].dropna().unique()
    
    for value in segment_values:
        print("\n" + "="*80)
        print(f"SEGMENT: {segment_name} = {value}")
        print("="*80)
        
        segment_df = df[df[segment_col] == value]
        
        if len(segment_df) < 20:  
            print(f"Skipping - insufficient sample size ({len(segment_df)} users)")
            continue
        
        try:
            results[f"{segment_name}_{value}"] = analyze_ab_test(segment_df, variant_column)
        except Exception as e:
            print(f"Error analyzing segment {value}: {e}")
    
    return results

In [7]:
def create_full_analysis(df: pd.DataFrame, variant_column: str = 'split') -> Dict[str, pd.DataFrame]:
    segments = {}
    
    print("\n" + "="*80)
    print("OVERALL AB TEST ANALYSIS")
    print("="*80)
    segments['overall'] = analyze_ab_test(df, variant_column)
    
    subscription_segments = analyze_by_segment(df, 'subscription', 'Subscription', variant_column)
    segments.update(subscription_segments)
    
    utm_segments = analyze_by_segment(df, 'utm_source', 'UTM_Source', variant_column)
    segments.update(utm_segments)
    
    geo_segments = analyze_by_segment(df, 'geo', 'Geo', variant_column)
    segments.update(geo_segments)
    
    payment_segments = analyze_by_segment(df, 'payment_method', 'Payment_Method', variant_column)
    segments.update(payment_segments)
    
    channel_segments = analyze_by_segment(df, 'channel', 'Channel', variant_column)
    segments.update(channel_segments)
    
    return segments

In [8]:
def generate_summary_report(segments: Dict[str, pd.DataFrame], output_file: str = 'upsell_11_15-3-1_15-4-1_ab_test_new_summary.txt'):
    negative_metrics = {
        'Unsub 12h Rate (on View)',
        'Unsub 12h Rate (on Upsell Purchase)',
        'Ticket Users Share (on All Users)',
        'Ticket Users Share (on Purch Users)'
    }

    with open(output_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("UPSELL A/B TEST ANALYSIS SUMMARY REPORT\n")
        f.write("="*80 + "\n\n")
        
        for segment_name, results_df in segments.items():
            f.write(f"\n{segment_name.upper()}\n")
            f.write("-"*80 + "\n")
            
            try:
                sig_results = results_df[results_df['significant'] == 'YES']
            except:
                sig_results = []
            
            if len(sig_results) > 0:
                f.write(f"Found {len(sig_results)} significant differences:\n\n")
                
                for _, row in results_df.iterrows():
                    
                    direction = row['direction']

                    # инвертируем направление для "плохих" метрик
                    if row['metric'] in negative_metrics:
                        if direction == 'POSITIVE':
                            direction = 'NEGATIVE'
                        elif direction == 'NEGATIVE':
                            direction = 'POSITIVE'

                    f.write(f"  • {row['metric']}: {direction} difference\n")
                    f.write(f"    clean: {row['clean_value']}, test: {row['test_value']}\n")
                    f.write(f"    Abs change: {row['absolute_difference']}\n")
                    f.write(f"    Relative change: {row['relative_difference_%']}%\n")
                    f.write(f"    p-value: {row['p_value']}\n")
                    f.write(f"    Significant: {row['significant']}\n\n")
            
            else:
                f.write("No significant differences found.\n\n")
    
    print(f"\n✓ Summary report saved to '{output_file}'")

In [9]:
df = pd.read_csv('test11_2.csv')

In [10]:
df['ltv'] = df['ltv'].fillna(0)
df['ticket_count'] = df['ticket_count'].fillna(0)

In [11]:
df[df['unsub12h'] == 1]['user_id'].nunique() / df[df['ups_purched'] == 1]['user_id'].nunique() 

0.045454545454545456

In [12]:
df['is_ticket'] = df.apply(lambda x: 1 if x['ticket_count'] != 0 else 0, axis = 1)

In [13]:
user_df = convert_to_user_level(df)

In [14]:
segments = create_full_analysis(user_df)


OVERALL AB TEST ANALYSIS
clean: u15.3.1
test: u15.4.1
clean n: 107
test n: 190


SEGMENT: Subscription = 4Week
clean: u15.3.1
test: u15.4.1
clean n: 78
test n: 116


SEGMENT: Subscription = 12Week
clean: u15.3.1
test: u15.4.1
clean n: 29
test n: 74


SEGMENT: UTM_Source = facebook
clean: u15.3.1
test: u15.4.1
clean n: 55
test n: 124


SEGMENT: UTM_Source = google
clean: u15.3.1
test: u15.4.1
clean n: 43
test n: 49


SEGMENT: UTM_Source = fb_page
Skipping - insufficient sample size (5 users)

SEGMENT: UTM_Source = landing_framer
Skipping - insufficient sample size (12 users)

SEGMENT: Geo = T1
clean: u15.3.1
test: u15.4.1
clean n: 106
test n: 188


SEGMENT: Geo = WW
Skipping - insufficient sample size (3 users)

SEGMENT: Payment_Method = paypal-vault
clean: u15.3.1
test: u15.4.1
clean n: 107
test n: 190


SEGMENT: Channel = solidgate
clean: u15.3.1
test: u15.4.1
clean n: 107
test n: 190



In [15]:
generate_summary_report(segments)


✓ Summary report saved to 'upsell_11_15-3-1_15-4-1_ab_test_new_summary.txt'


In [23]:
import numpy as np
import scipy.stats as sps
from typing import Union
import pandas as pd
from google.cloud import bigquery

In [24]:
c_mean = user_df[user_df['split']=="u15.3.1"]["purch_amount"].mean()
t_mean = user_df[user_df['split']=="u15.4.1"]["purch_amount"].mean()
c_std = user_df[user_df['split']=="u15.3.1"]["purch_amount"].std()
t_std = user_df[user_df['split']=="u15.4.1"]["purch_amount"].std()

In [32]:
def get_sample_size(var_x, var_y=None, mde=5,  alpha=0.05, beta=0.8, alternative='two-sided'):
    """
    Calculate the Minimum Sample Size for given MDE.

    Args:
        var_x (float): Variance of group X.
        var_y (float): Variance of group Y.
        mde (float): MDE.
        alternative (str): Specifies the alternative hypothesis. Options are 'two-sided', 'less', or 'greater'.
        alpha (float): Significance level.
        beta (float): Power.

    Returns:
        int. Sample size for a given MDE
    """
    if not var_y:
        var_y = var_x
    if alternative == 'two-sided':
        z_alpha = sps.norm.ppf(1 - alpha / 2)  # Two-tailed z-score for alpha
    elif alternative in ['less', 'greater']:
        z_alpha = sps.norm.ppf(1 - alpha)     # One-tailed z-score for alpha
    else:
        raise ValueError(
            "Invalid alternative hypothesis. Choose from 'two-sided', 'less', or 'greater'.")

    z_beta = sps.norm.ppf(beta)
    n = (z_alpha + z_beta)**2 * (var_x + var_y) / (mde**2)
    return int(np.ceil(n))

print(get_sample_size(c_std**2, t_std**2, mde=8,  alpha=0.05, beta=0.8, alternative='greater'))

97


In [16]:
from scipy import stats
import numpy as np

def power_binary(p1, n, mde_abs, alpha=0.05, alternative='greater'):
    """
    Мощность для бинарной метрики (конверсии).
    p1: конверсия в контроле
    n: размер выборки на группу (количество пользователей в знаменателе)
    mde_abs: абсолютный прирост (например, 0.05 = +5 п.п.)
    alternative: 'greater' (тест на увеличение), 'less' (уменьшение), 'two-sided'
    """
    p2 = p1 + mde_abs

    if p2 == 1:
        se = np.sqrt(p1*(1-p1)/n)
        z_obs = (1 - p1) / se
    elif p2 == 0:
        se = np.sqrt(p1*(1-p1)/n)
        z_obs = (0 - p1) / se
    else:
        se = np.sqrt(p1*(1-p1)/n + p2*(1-p2)/n)
        z_obs = (p2 - p1) / se

    if alternative == 'greater':
        z_crit = stats.norm.ppf(1 - alpha)
        power = 1 - stats.norm.cdf(z_crit - z_obs)
    elif alternative == 'less':
        z_crit = stats.norm.ppf(alpha)
        power = stats.norm.cdf(z_crit - z_obs)
    else:  # two-sided
        z_crit = stats.norm.ppf(1 - alpha/2)
        power = (1 - stats.norm.cdf(z_crit - z_obs)) + stats.norm.cdf(-z_crit - z_obs)
    return power

def mde_binary(p1, n, target_power=0.8, alpha=0.05, alternative='greater'):
    """
    Минимальный детектируемый абсолютный эффект (MDE) для бинарной метрики.
    """
    from scipy.optimize import brentq
    max_mde = 1 - p1
    power_at_max = power_binary(p1, n, max_mde, alpha, alternative)
    if power_at_max < target_power:
        print(f"  Предупреждение: даже при максимальном MDE = {max_mde:.2%} мощность = {power_at_max:.2%} < {target_power:.0%}")
        return float('inf')
    def f(mde):
        return power_binary(p1, n, mde, alpha, alternative) - target_power
    try:
        mde = brentq(f, 0, max_mde)
        return mde
    except ValueError:
        return np.nan

def power_continuous(mu1, sigma, n, mde_abs, alpha=0.05, alternative='greater'):
    """
    Мощность для непрерывной метрики (среднее).
    mu1: среднее в контроле
    sigma: стандартное отклонение (можно взять из контрола)
    n: размер выборки на группу
    mde_abs: абсолютное изменение среднего
    """
    se = sigma * np.sqrt(2/n)
    z_obs = mde_abs / se
    if alternative == 'greater':
        z_crit = stats.norm.ppf(1 - alpha)
        power = 1 - stats.norm.cdf(z_crit - z_obs)
    elif alternative == 'less':
        z_crit = stats.norm.ppf(alpha)
        power = stats.norm.cdf(z_crit - z_obs)
    else:  # two-sided
        z_crit = stats.norm.ppf(1 - alpha/2)
        power = (1 - stats.norm.cdf(z_crit - z_obs)) + stats.norm.cdf(-z_crit - z_obs)
    return power

def mde_continuous(mu1, sigma, n, target_power=0.8, alpha=0.05, alternative='greater'):
    """
    Минимальный детектируемый абсолютный эффект (MDE) для непрерывной метрики.
    """
    from scipy.optimize import brentq
    def f(mde):
        return power_continuous(mu1, sigma, n, mde, alpha, alternative) - target_power
    try:
        mde = brentq(f, 0, 1e6)  
        return mde
    except ValueError:
        return np.nan


def estimate_metric_sensitivity(clean_df, test_df, metric_config, alpha=0.05, target_power=0.8):
    """
    Оценивает мощность для заданного MDE или MDE для целевой мощности для одной метрики.
    metric_config: dict с ключами:
        - 'name': название метрики (для вывода)
        - 'type': 'binary' или 'continuous'
        - 'denominator_col': столбец, определяющий популяцию (если None, используется вся группа)
        - 'value_col': для бинарных – столбец-индикатор успеха; для непрерывных – столбец значений
        - 'alternative': 'greater' или 'less'
        - 'mde_abs' (опционально): если задан, рассчитывается мощность для этого MDE
        - Если не задан, рассчитывается MDE для target_power
    """
    if metric_config.get('denominator_col') is None:
        clean_pop = clean_df
        test_pop = test_df
    else:

        clean_pop = clean_df[clean_df[metric_config['denominator_col']] == 1]
        test_pop = test_df[test_df[metric_config['denominator_col']] == 1]
    n_clean = len(clean_pop)
    n_test = len(test_pop)
    if n_clean == 0 or n_test == 0:
        return {'error': 'No users in denominator'}

    # Вычисляем базовые статистики
    if metric_config['type'] == 'binary':
        p1 = clean_pop[metric_config['value_col']].mean()
        if 'mde_abs' in metric_config:
            power = power_binary(p1, n_clean, metric_config['mde_abs'],
                                 alpha=alpha, alternative=metric_config.get('alternative','greater'))
            return {
                'metric': metric_config['name'],
                'n_per_group': n_clean,
                'baseline_rate': p1,
                'mde_abs': metric_config['mde_abs'],
                'power': power
            }
        else:
            mde = mde_binary(p1, n_clean, target_power=target_power,
                             alpha=alpha, alternative=metric_config.get('alternative','greater'))
            return {
                'metric': metric_config['name'],
                'n_per_group': n_clean,
                'baseline_rate': p1,
                'mde_abs_for_power': mde,
                'target_power': target_power
            }
    else:  # continuous
        mu1 = clean_pop[metric_config['value_col']].mean()
        sigma = clean_pop[metric_config['value_col']].std()
        if 'mde_abs' in metric_config:
            power = power_continuous(mu1, sigma, n_clean, metric_config['mde_abs'],
                                     alpha=alpha, alternative=metric_config.get('alternative','greater'))
            return {
                'metric': metric_config['name'],
                'n_per_group': n_clean,
                'baseline_mean': mu1,
                'baseline_std': sigma,
                'mde_abs': metric_config['mde_abs'],
                'power': power
            }
        else:
            mde = mde_continuous(mu1, sigma, n_clean, target_power=target_power,
                                 alpha=alpha, alternative=metric_config.get('alternative','greater'))
            return {
                'metric': metric_config['name'],
                'n_per_group': n_clean,
                'baseline_mean': mu1,
                'baseline_std': sigma,
                'mde_abs_for_power': mde,
                'target_power': target_power
            }

clean_variant = sorted(user_df['split'].unique())[0]
test_variant = sorted(user_df['split'].unique())[1]
clean_df = user_df[user_df['split'] == clean_variant].copy()
test_df = user_df[user_df['split'] == test_variant].copy()

metrics_to_analyze = [
    {
        'name': 'Upsell Rate (on View)',
        'type': 'binary',
        'denominator_col': 'ups_view',
        'value_col': 'ups_purched',
        'alternative': 'greater'
    },
    {
        'name': 'Upsell TTP Rate (on View)',
        'type': 'binary',
        'denominator_col': 'ups_view',
        'value_col': 'ups_ttp',
        'alternative': 'greater'
    },
    {
        'name': 'Upsell SR (on TTP)',
        'type': 'binary',
        'denominator_col': 'ups_ttp',
        'value_col': 'ups_purched',
        'alternative': 'greater'
    },
    {
        'name': 'Unsub 12h Rate (on View)',
        'type': 'binary',
        'denominator_col': 'ups_view',
        'value_col': 'unsub12h',
        'alternative': 'less'  
    },
    {
        'name': 'Unsub 12h Rate (on Upsell Purchase)',
        'type': 'binary',
        'denominator_col': 'ups_purched',
        'value_col': 'unsub12h',
        'alternative': 'less'
    },
    {
        'name': 'Ticket Users Share (on All Users)',
        'type': 'binary',
        'denominator_col': None,
        'value_col': 'is_ticket',
        'alternative': 'greater' 
    },
    {
        'name': 'Ticket Users Share (on Purch Users)',
        'type': 'binary',
        'denominator_col': 'ups_purched',
        'value_col': 'is_ticket',
        'alternative': 'greater'
    },
    {
        'name': 'Upsell Gain (on View)',
        'type': 'continuous',
        'denominator_col': 'ups_view',
        'value_col': 'purch_amount',
        'alternative': 'greater'
    },
    {
        'name': 'LTV',
        'type': 'continuous',
        'denominator_col': 'ups_view',
        'value_col': 'ltv',
        'alternative': 'greater'
    },
    {
        'name': 'AOV (on Purch Users)',
        'type': 'continuous',
        'denominator_col': 'ups_purched',
        'value_col': 'purch_amount',
        'alternative': 'greater'
    }
]

print("="*80)
print("SENSITIVITY ANALYSIS: Minimal Detectable Effect (MDE) for target power = 80%")
print("="*80)
for cfg in metrics_to_analyze:
    res = estimate_metric_sensitivity(clean_df, test_df, cfg, target_power=0.8)
    if 'error' in res:
        print(f"{res['metric']}: {res['error']}")
    else:
        if cfg['type'] == 'binary':
            if 'mde_abs_for_power' in res:
                mde_val = res['mde_abs_for_power']
                if np.isinf(mde_val):
                    mde_str = ">100% (недостижимо)"
                else:
                    mde_str = f"{mde_val:.2%}"
                print(f"{res['metric']}: n={res['n_per_group']}, baseline={res['baseline_rate']:.2%}, MDE(80% power) = {mde_str}")
            else:
                print(f"{res['metric']}: n={res['n_per_group']}, baseline={res['baseline_rate']:.2%}, power for MDE={res['mde_abs']:.2%} = {res['power']:.2%}")
        else:
            if 'mde_abs_for_power' in res:
                mde_val = res['mde_abs_for_power']
                print(f"{res['metric']}: n={res['n_per_group']}, mean={res['baseline_mean']:.2f}, std={res['baseline_std']:.2f}, MDE(80% power) = {mde_val:.2f}")
            else:
                print(f"{res['metric']}: n={res['n_per_group']}, mean={res['baseline_mean']:.2f}, std={res['baseline_std']:.2f}, power for MDE={res['mde_abs']:.2f} = {res['power']:.2%}")

print("\n" + "="*80)
print("SENSITIVITY ANALYSIS: Power for a given MDE (e.g., +5% absolute for binary, +10% relative for continuous)")
print("="*80)
for cfg in metrics_to_analyze:
    if cfg['type'] == 'binary':
        mde_abs = 0.05
    else:
        if cfg.get('denominator_col') is None:
            clean_pop = clean_df
        else:
            clean_pop = clean_df[clean_df[cfg['denominator_col']] == 1]
        if len(clean_pop) == 0:
            continue
        mu1 = clean_pop[cfg['value_col']].mean()
        mde_abs = mu1 * 0.10 
    cfg_with_mde = cfg.copy()
    cfg_with_mde['mde_abs'] = mde_abs
    res = estimate_metric_sensitivity(clean_df, test_df, cfg_with_mde)
    if 'error' in res:
        print(f"{res.get('metric', cfg['name'])}: {res['error']}")
    else:
        if cfg['type'] == 'binary':
            print(f"{res['metric']}: n={res['n_per_group']}, baseline={res['baseline_rate']:.2%}, MDE={res['mde_abs']:.2%} => Power = {res['power']:.2%}")
        else:
            print(f"{res['metric']}: n={res['n_per_group']}, mean={res['baseline_mean']:.2f}, MDE={res['mde_abs']:.2f} => Power = {res['power']:.2%}")

SENSITIVITY ANALYSIS: Minimal Detectable Effect (MDE) for target power = 80%
Upsell Rate (on View): n=107, baseline=18.69%, MDE(80% power) = 14.71%
Upsell TTP Rate (on View): n=107, baseline=18.69%, MDE(80% power) = 14.71%
Upsell SR (on TTP): n=20, baseline=100.00%, MDE(80% power) = nan%
  Предупреждение: даже при максимальном MDE = 99.07% мощность = 0.00% < 80%
Unsub 12h Rate (on View): n=107, baseline=0.93%, MDE(80% power) = >100% (недостижимо)
  Предупреждение: даже при максимальном MDE = 95.00% мощность = 0.00% < 80%
Unsub 12h Rate (on Upsell Purchase): n=20, baseline=5.00%, MDE(80% power) = >100% (недостижимо)
Ticket Users Share (on All Users): n=107, baseline=0.00%, MDE(80% power) = nan%
Ticket Users Share (on Purch Users): n=20, baseline=0.00%, MDE(80% power) = nan%
Upsell Gain (on View): n=107, mean=10.35, std=23.13, MDE(80% power) = 7.86
LTV: n=107, mean=102.19, std=50.58, MDE(80% power) = 17.19
AOV (on Purch Users): n=20, mean=55.38, std=18.95, MDE(80% power) = 14.90

SENSITI